<a href="https://colab.research.google.com/github/jdasam/mas1004-assignment1/blob/main/assignment1_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 1 on Google Colab

Colab is where you run this assignment on a GPU: downloading images, training,
cleaning and checking. You do not write code here, and you do not publish the
page from here.

- You write your functions on your own computer, with your coding agent, and
  push them to your repository on GitHub. This notebook brings the code over
  from there, in section 3 and again in section 9 every time you push. Do not
  edit code in Colab. The next update replaces it with what is on GitHub.
- When training is done, you take the model files to your own computer and
  publish the page from there (section 15).
- If your computer cannot run the assignment at all (an Intel Mac, or Windows
  on an ARM processor, see README), your agent still writes the code on your
  computer, and you run the tests here after each push.

Two more things about Colab will catch you out, so deal with both before you
start:

1. Everything you save on the Colab machine is deleted when your session ends.
   Your downloaded images would go with it. So we keep the whole assignment in
   your Google Drive instead, and Colab just works on it there.
2. It gives you a plain CPU unless you ask for a GPU.

Turn the GPU on. Training ResNet18 on a CPU takes several minutes per run, and
on a GPU it is much faster. The menu at the top, `Runtime` > `Change runtime
type`, pick `T4 GPU`, then `Save`. The session restarts, so do that first.
The free GPU has a daily limit, so turn it back off when you are only looking
at pictures or writing.

## 1. What machine did we get?

In [ ]:
import torch, platform
print("python  ", platform.python_version())
print("torch   ", torch.__version__)
print("GPU     ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none, running on CPU")

## 2. Connect your Google Drive

A box will pop up asking you to sign in and allow access. It is asking for your
own Drive, so that the images you download survive after this session ends.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

ValueError: mount failed

In [3]:
from pathlib import Path
import os
import subprocess

PROJECT = Path("/content/mas1004-assignment1")

if not PROJECT.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/hy743/mas1004-assignment1.git", str(PROJECT)],
        check=True
    )

os.chdir(PROJECT)
subprocess.run(
    ["pip", "install", "-q", "-r", "requirements.txt"],
    check=True
)
subprocess.run(
    ["python", "-m", "pytest", "-q", "tests/test_export.py",
     "tests/test_data.py", "tests/test_train.py", "tests/test_evaluate.py"],
    check=True
)

CompletedProcess(args=['python', '-m', 'pytest', '-q', 'tests/test_export.py', 'tests/test_data.py', 'tests/test_train.py', 'tests/test_evaluate.py'], returncode=0)

In [4]:
import os
import subprocess
import shutil
from pathlib import Path

os.chdir("/content/mas1004-assignment1")

subprocess.run([
    "python", "src/collect.py",
    "--classes", "squat exercise,lunge exercise,push up exercise",
    "--n", "150"
], check=True)

raw = Path("data/raw")
clean = Path("data/clean")
if not clean.exists():
    shutil.copytree(raw, clean)

for folder in sorted(clean.iterdir()):
    if folder.is_dir():
        print(folder.name, len(list(folder.iterdir())), "files")

lunge_exercise 150 files
push_up_exercise 150 files
squat_exercise 150 files


In [5]:
import os
import subprocess
import torch

os.chdir("/content/mas1004-assignment1")
if not torch.cuda.is_available():
    raise RuntimeError("GPU가 연결되지 않았어요. 런타임에서 T4 GPU를 선택해주세요.")

runs = [
    ["--tag", "run"],
    ["--scratch", "--tag", "scratch"],
    ["--freeze", "--lr", "1e-3", "--tag", "frozen"],
    ["--lr", "3e-4", "--tag", "higher_lr"],
]

for options in runs:
    subprocess.run([
        "python", "-u", "src/run.py",
        "--epochs", "3", "--batch-size", "32",
        *options
    ], check=True)

print("학습 4회 완료")

학습 4회 완료


In [6]:
import os
import subprocess
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

os.chdir("/content/mas1004-assignment1")

subprocess.run(["python", "src/clean.py", "look"], check=True)

with ZipFile("/content/review.zip", "w", ZIP_DEFLATED) as z:
    for p in Path("results").rglob("*"):
        if p.is_file() and p.suffix in {".txt", ".json", ".png", ".csv"}:
            z.write(p, str(p))

files.download("/content/review.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
import subprocess
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

result = subprocess.run(
    ["python", "-u", "src/clean.py", "suspects"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)
print(result.stdout)
Path("results/suspects_output.txt").write_text(result.stdout)
result.check_returncode()

with ZipFile("/content/suspects.zip", "w", ZIP_DEFLATED) as z:
    for p in Path("results/cleaning/suspects").rglob("*"):
        if p.is_file():
            z.write(p, str(p))
    z.write("results/suspects_output.txt")

files.download("/content/suspects.zip")

describing 450 images with ResNet18 ...
  32/450
  64/450
  96/450
  128/450
  160/450
  192/450
  224/450
  256/450
  288/450
  320/450
  352/450
  384/450
  416/450
  448/450
  450/450
wrote results/cleaning/suspects/lunge_exercise.png
wrote results/cleaning/suspects/push_up_exercise.png
wrote results/cleaning/suspects/squat_exercise.png
wrote results/cleaning/suspects/copies.png
wrote results/cleaning/suspects/suspects.csv

Agreement with the label, over all 450 images: 86.2% would be labelled the same by the rest of your data.
Look at each suspect and decide by your rule. Some are fine pictures that are just unusual, and those are worth keeping.



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
import subprocess

subprocess.run([
    "python", "-u", "src/collect.py",
    "--classes", "push-up workout proper form",
    "--n", "100",
    "--out", "data/replacement_raw"
], check=True)

print("푸시업 재수집 완료")

import os
import subprocess

os.chdir("/content/mas1004-assignment1")

r = subprocess.run([
    "python", "-u", "src/collect.py",
    "--classes", "push-up workout proper form",
    "--n", "100",
    "--out", "data/replacement_raw"
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

print(r.stdout)
print("종료 코드:", r.returncode)

CalledProcessError: Command '['python', '-u', 'src/collect.py', '--classes', 'push-up workout proper form', '--n', '100', '--out', 'data/replacement_raw']' returned non-zero exit status 1.

In [13]:
import shutil
import subprocess
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

old = Path("data/clean/push_up_exercise")
backup = Path("data/backup_original_push_up")

if old.exists() and not backup.exists():
    shutil.move(str(old), str(backup))

old.mkdir(parents=True, exist_ok=True)
source = Path("data/replacement_raw/push-up_workout_proper_form")
for p in sorted(source.glob("*.jpg"))[:25]:
    shutil.copy2(p, old / p.name)

for command, log in [
    (["python", "-u", "src/run.py", "--tag", "clean",
      "--epochs", "3", "--lr", "1e-4"], "results/clean_execution.txt"),
    (["python", "-u", "src/export_web.py", "--tag", "clean"],
     "results/export_output.txt"),
]:
    with open(log, "w") as f:
        process = subprocess.Popen(
            command, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True
        )
        for line in process.stdout:
            print(line, end="")
            f.write(line)
    if process.wait() != 0:
        raise RuntimeError(f"실행 실패: {log}")

with ZipFile("/content/demo_results.zip", "w", ZIP_DEFLATED) as z:
    for folder in ["docs", "results"]:
        for p in Path(folder).rglob("*"):
            if p.is_file() and p.suffix != ".pt":
                z.write(p, str(p))

files.download("/content/demo_results.zip")

reading /content/mas1004-assignment1/data/clean
325 images, 3 classes, read in 7 s
  lunge_exercise           150
  push_up_exercise         25
  squat_exercise           150
training on 260, testing on 65
model has 11,178,051 parameters, 11,178,051 of them will be trained
epoch 1/3: train_loss=0.862934 train_acc=0.615385 test_loss=0.752810 test_acc=0.630769
epoch 2/3: train_loss=0.210173 train_acc=0.973077 test_loss=0.487446 test_acc=0.815385
epoch 3/3: train_loss=0.089143 train_acc=0.996154 test_loss=0.415041 test_acc=0.800000

trained in 0.1 minutes
train accuracy 100.0%
test  accuracy 80.0%
wrote /content/mas1004-assignment1/results/clean_curves.png
wrote /content/mas1004-assignment1/results/clean_confusion.png
wrote /content/mas1004-assignment1/results/clean_worst.png
wrote /content/mas1004-assignment1/results/clean_model.pt

One line for your experiment table:
| clean | ImageNet | all layers | 3 | 0.0001 | 11,178,051 | 100.0% | 80.0% |

To put this model on your web page: python 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
import os
import re
import subprocess
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

os.chdir("/content/mas1004-assignment1")
# 이미 업로드한 사진 사용

number_classes = {
    3: "squat_exercise",
    4: "squat_exercise",
    5: "squat_exercise",
    6: "lunge_exercise",
    7: "lunge_exercise",
    8: "push_up_exercise",
    9: "push_up_exercise",
}

for filename, content in uploaded.items():
    name = re.sub(r"\s*\(\d+\)(?=\.jpg$)", "", filename, flags=re.I).strip()
    lower = name.lower()
    match = re.fullmatch(r"-(\d+)\.jpg", lower)

    if match:
        category = number_classes.get(int(match.group(1)))
    elif "squat" in lower or "treino" in lower:
        category = "squat_exercise"
    elif any(s in lower for s in ["9 best", "elliot", "strength training"]):
        category = "lunge_exercise"
    elif any(s in lower for s in ["close-grip", "fitness", "grip exercises"]):
        category = "push_up_exercise"
    else:
        raise ValueError(f"사진 파일을 선택해주세요: {filename}")

    if category is None:
        raise ValueError(f"분류할 수 없는 파일: {filename}")

    destination = Path("data/new_images") / category / name
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(content)

for category in ["lunge_exercise", "push_up_exercise", "squat_exercise"]:
    folder = Path("data/new_images") / category
    count = len(list(folder.glob("*.jpg")))
    print(category, count, "장")
    if count < 5:
        raise ValueError(f"{category}: 사진이 부족해요. 15장을 모두 선택해주세요.")

Path("results").mkdir(exist_ok=True)

commands = [
    (["python", "-u", "src/clean.py", "overlap"], "overlap.txt"),
    (["python", "-u", "src/check.py", "--compare", "run", "clean"], "compare.txt"),
    (["python", "-u", "src/check.py"], "check.txt"),
]

for command, filename in commands:
    result = subprocess.run(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )
    print(result.stdout)
    Path("results", filename).write_text(result.stdout)

with ZipFile("/content/evaluation.zip", "w", ZIP_DEFLATED) as z:
    for folder in ["results", "data/new_images"]:
        for p in Path(folder).rglob("*"):
            if p.is_file() and p.suffix != ".pt":
                z.write(p, str(p))

files.download("/content/evaluation.zip")

lunge_exercise 5 장
push_up_exercise 5 장
squat_exercise 5 장
describing 40 images with ResNet18 ...
  32/40
  40/40
none of your 15 new images is a near copy of any of the 474 images you downloaded

the closest any new image comes to a downloaded one: 0.95 (near copies are above 0.95)

MAS1004 Assignment 1, checking your work

Training images  (/content/mas1004-assignment1/data/clean)
----------------------------------------------------------
  lunge_exercise                 150 images
  push_up_exercise                25 images
  squat_exercise                 150 images
  warning  "push_up_exercise" has only 25 images. With this few, the model has little to learn that class from, and its test accuracy rests on a handful of test images.
  warning  the biggest class has 150 images and the smallest has 25. A model can score well just by always guessing the big class.

Your images from a new source  (/content/mas1004-assignment1/data/new_images)
--------------------------------------------

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [19]:
!printf 'y\n\n' | GH_BROWSER=echo gh auth login --hostname github.com --git-protocol https --web

! Did you know that expiring tokens can be requested with --expiring? Read more at https://gh.io/gh-expiring-tokens

! Failed to copy one-time code to clipboard
  No clipboard utilities available. Please install xsel, xclip, wl-clipboard or Termux:API add-on for termux-clipboard-get/set.
! First copy your one-time code: 9E06-CB76
Open this URL to continue in your web browser: https://github.com/login/device
✓ Authentication complete.
- gh config set -h github.com git_protocol https
✓ Configured git protocol
! Authentication credentials saved in plain text
✓ Logged in as hy743


In [20]:
import os
import json
import subprocess

os.chdir("/content/mas1004-assignment1")

def run(args, **kwargs):
    return subprocess.run(args, check=True, text=True, **kwargs)

run(["gh", "auth", "setup-git"])
run(["git", "config", "user.name", "hy743"])
run(["git", "config", "user.email", "hy743@users.noreply.github.com"])
run(["git", "pull", "--ff-only", "origin", "main"])
run(["git", "add",
     "docs/model.onnx", "docs/model.json", "docs/selftest.json"])

changes = subprocess.run(
    ["git", "diff", "--cached", "--quiet"]
).returncode

if changes == 1:
    run(["git", "commit", "-m", "Publish trained exercise classifier"])

run(["git", "push", "origin", "main"])

endpoint = "repos/hy743/mas1004-assignment1/pages"
existing = subprocess.run(
    ["gh", "api", endpoint],
    capture_output=True, text=True
)

if existing.returncode != 0:
    run(
        ["gh", "api", "--method", "POST", endpoint, "--input", "-"],
        input=json.dumps({
            "source": {"branch": "main", "path": "/docs"}
        })
    )

run(["gh", "api", endpoint, "--jq", ".html_url"])
print("업로드와 Pages 설정 완료")

업로드와 Pages 설정 완료


## 3. Bring your repository into your Drive

Put the address of your own repository in `REPO` below, the copy your agent
made at the start (README, Getting started). Not the template itself.

The first time, this copies your repository into `My Drive/mas1004/assignment1`.
After that, it makes the code there the same as your repository on GitHub now,
and prints which commit that is. Your images in `data/` and your `results/` are
not in git, so this never touches them. Anything else you changed in that
folder by hand is replaced.

If that folder was made from another address, for example from the template in
class on 29 September, it is renamed to `assignment1_old_...` and your
repository is copied in fresh. The `data/` and `results/` folders of the old
one are moved into the new one, so you keep your images.

If it stops with "could not read Username", your repository is private. Make it
public on GitHub, under Settings, and run the cell again.

In [ ]:
import os
import shutil
import subprocess
import time
from pathlib import Path

# Your own repository, for example "https://github.com/hong-gildong/my-classifier"
REPO = "https://github.com/hy743/mas1004-assignment1"

TEMPLATE = "https://github.com/jdasam/mas1004-assignment1"
HOME = Path("/content/drive/MyDrive/mas1004")
PROJECT = HOME / "assignment1"


def git(*args):
    done = subprocess.run(["git", *args], capture_output=True, text=True)
    if done.returncode != 0:
        raise RuntimeError(f"git {' '.join(args)} failed:\n{done.stderr}")
    return done.stdout


def address(url):
    return url.strip().rstrip("/").removesuffix(".git").lower()


def bring_code():
    """Make the code in PROJECT the same as your repository on GitHub."""
    git("-C", str(PROJECT), "fetch", "-q")
    git("-C", str(PROJECT), "reset", "-q", "--hard", "@{u}")
    print(git("-C", str(PROJECT), "log", "-1",
              "--format=code from your commit %h (%cr): %s"))


if "YOUR-NAME" in REPO or address(REPO) == address(TEMPLATE):
    raise ValueError("Put the address of your own repository in REPO, "
                     "then run this cell again.")

# git refuses to work in a folder it thinks belongs to another user, and folders
# in Drive can look like that.
git("config", "--global", "--add", "safe.directory", "*")

HOME.mkdir(parents=True, exist_ok=True)
old = None
if PROJECT.exists():
    try:
        here = address(git("-C", str(PROJECT), "remote", "get-url", "origin"))
    except RuntimeError:
        here = None
    if here != address(REPO):
        old = HOME / time.strftime("assignment1_old_%m%d_%H%M")
        PROJECT.rename(old)
        print(f"{PROJECT} came from {here}, so it is now {old}")

if not PROJECT.exists():
    git("clone", "-q", REPO.strip(), str(PROJECT))
    print(f"copied {REPO.strip()} into {PROJECT}")
    for name in ("data", "results"):
        if old is not None and (old / name).exists():
            shutil.move(str(old / name), str(PROJECT / name))
            print(f"moved {name}/ over from {old}")

bring_code()
os.chdir(PROJECT)
print("working in", Path.cwd())

## 4. Install what Colab does not already have

PyTorch is already there. This adds the image downloader and the ONNX tools
that put your model on the web page. Colab forgets them when the session ends,
so run this every time you open the notebook.

In [ ]:
!pip install -q -r requirements.txt

## 5. Check that the given code works

Six tests should pass. These cover code you were given, not code you write.

In [ ]:
!python -m pytest -q tests/test_export.py

## 6. Download your images

Change the search phrases to your own categories. This writes into
`data/raw` inside your Drive, so it is still there tomorrow.

If you already downloaded images on your own computer, do not download them
again here. Upload your `data` folder into `My Drive/mas1004/assignment1/` in
Google Drive instead, so that you keep the same images. Either way, from here
on your images live in Drive, and you clean them here, in section 11.

In [ ]:
!python src/collect.py --classes "squat exercise,lunge exercise,push up exercise" --n 150
if not Path("data/clean").exists():
    !cp -r data/raw data/clean

## 7. Look at every image

This draws every image onto sheets. Look at all of them. Do not remove anything
yet: your first training run is on the images as they came.

In [ ]:
!python src/clean.py look
from IPython.display import Image, display
for sheet in sorted(Path("results/cleaning/look").glob("*.png")):
    print(sheet.name)
    display(Image(filename=str(sheet)))

## 8. Put your images from a new source in

Make a folder `data/new_images/<category>` for each category, with exactly the
same names as the folders in `data/clean`, and put at least five images into
each. They must come from a source you can be sure is not in your downloads,
such as photos you took yourself or a friend took, or frames from a video you
recorded, or images from the web that you are sure your download could not
have found. Save them as JPEG or PNG: the code here cannot read the HEIC files
an iPhone makes by default. The file browser on the left can upload
them, under `drive/MyDrive/mas1004/assignment1/data`, or you can drop them
into the same folder in Google Drive from your phone.

The second cell checks that none of them is a near copy of a downloaded image.

In [ ]:
for folder in sorted(Path("data/clean").iterdir()):
    mine = Path("data/new_images") / folder.name
    mine.mkdir(parents=True, exist_ok=True)
    print(f"{mine}   {len(list(mine.iterdir()))} images")

In [ ]:
!python src/clean.py overlap | tee results/overlap.txt
if Path("results/cleaning/overlap.png").exists():
    display(Image(filename="results/cleaning/overlap.png"))

## 9. Bring in your code and test it

Write the functions in `src/data.py`, `src/train.py` and `src/evaluate.py` on
your own computer, with your agent, and run the tests there. Then commit and
push. The first cell below brings what you pushed into your Drive and prints
which commit it is. Check that it is the one you just pushed.

Run it again every time you push a change. Do not edit the code here: the next
time it runs, anything changed here is replaced with what is on GitHub.

The second cell runs the tests here too, so you can see that Colab has the same
code. The first run downloads the ImageNet weights. If your computer cannot run
the assignment, this is the only place you run the tests.

In [ ]:
# bring_code comes from the cell in section 3. If Colab says it is not defined,
# run sections 2 and 3 again first.
bring_code()

In [ ]:
!python -m pytest -q tests

## 10. Train, before cleaning anything

In [ ]:
!python src/run.py
!python src/run.py --scratch --tag scratch
!python src/run.py --freeze --lr 1e-3 --tag frozen

In [ ]:
for name in ["run_curves.png", "run_confusion.png", "run_worst.png"]:
    if Path(f"results/{name}").exists():
        print(name)
        display(Image(filename=f"results/{name}"))

## 11. Clean your data

Write your rule first. Then ask the rest of your data which images look least
like their class, and which are near copies of each other.

In [ ]:
!python src/clean.py suspects
for sheet in sorted(Path("results/cleaning/suspects").glob("*.png")):
    print(sheet.name)
    display(Image(filename=str(sheet)))

Remove what breaks your rule, with the reason. Repeat this cell as many times
as you need, with your own file names and reasons.

In [ ]:
!python src/clean.py remove wine_glass/0063.jpg wine_glass/0069.jpg --reason "chart, not a photo"

In [ ]:
!python src/clean.py count | tee results/clean_count.txt

## 12. Train on the clean data and compare on your new images

In [ ]:
!python src/run.py --tag clean
!python src/check.py --compare run clean | tee results/compare.txt

## 13. Export the clean model and check your work

This writes the model the page needs into `docs/` and runs the checker. Copy
the whole output of `check.py` into your long report.

`check.py` warns that `docs/model.onnx` is not committed. That is expected
here: you commit it on your own computer, in section 15.

In [ ]:
!python src/export_web.py --tag clean
!python src/check.py | tee results/check.txt

## 14. Try the demo page without leaving Colab

This serves the `docs/` folder and opens it in a panel. The upload and drawing
tabs work here. The webcam does not, because of how Colab sandboxes the page,
so test the webcam on your own machine or on your published page.

In [ ]:
import subprocess, time
from google.colab import output

server = subprocess.Popen(["python", "-m", "http.server", "-d", "docs", "8000"])
time.sleep(2)
output.serve_kernel_port_as_window(8000)

## 15. Take the model to your own computer and publish it there

Colab has done its part. The page is published from your own computer, not from
here.

The cell below packs what you need into `to_my_computer.zip` and downloads it:
the three files the page needs, from `docs/`, and the pictures, settings and
saved outputs in `results/` for your long report. The trained models themselves (`results/*.pt`,
45 MB each) are left out. If the download does not start, the zip is also in
your Drive, in `My Drive/mas1004/assignment1/`.

Then, on your own computer:

1. Unzip it. Copy `model.onnx`, `model.json` and `selftest.json` from its
   `docs` folder into the `docs` folder of your repository, next to
   `index.html` and `app.js`.
2. Look at the page with `uv run python -m http.server -d docs 8000` and
   http://localhost:8000, and check that the badge is green. If uv does not run
   on your computer, `python3 -m http.server -d docs 8000` does the same.
3. Commit and push, and turn on GitHub Pages, as in README, Problem 6.

The pictures from `results/` go into your report, not into git.

In [ ]:
!rm -f to_my_computer.zip
!zip -qr to_my_computer.zip docs/model.onnx docs/model.json docs/selftest.json results -x "results/*.pt" "results/cleaning/features.npz"
from google.colab import files
files.download("to_my_computer.zip")